# Custom Perceptron vs. Scikit-Learn SGDClassifier
This notebook demonstrates a comparison between a custom scratch-built Perceptron (incorporating random shuffling and convergence limits) and Scikit-Learn's `SGDClassifier` using perceptron loss.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

## 1. Dataset Generation and Preparation

In [ ]:
# Generate classification dataset
X, y = make_classification(n_samples=1000, n_features=2, n_informative=2, n_redundant=0, n_classes=2, random_state=42)
df = pd.DataFrame(X, columns=['x1', 'x2'])
df['target'] = y
display(df.head())

## 2. Train-Test Split and Scikit-Learn SGDClassifier (Perceptron Baseline)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[['x1', 'x2']], df['target'], test_size=0.2, random_state=42
)

# Initialize and train the baseline SGDClassifier
clf = SGDClassifier(loss='perceptron', penalty=None, random_state=42)
clf.fit(X_train, y_train)

# Evaluate SGDClassifier
y_pred = clf.predict(X_test)
print(f"SGDClassifier Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

## 3. Custom Perceptron Definition
Here is the custom implementation of the Perceptron algorithm, featuring optional randomized sample shuffling at the start of each training epoch.

In [ ]:
class CustomPerceptron:
  def __init__(self, dataset):
    self.out = dataset['target']
    self.inp = dataset.drop('target', axis=1)
    self.weights = np.array([-1.0] * len(self.inp.columns))

  def fit(self, epochs=100, lr=1e-2, shuffle=True, random_state=42):
    self.loss_history = []
    rng = np.random.default_rng(random_state)
    n_samples = len(self.inp)

    for i in range(epochs):
      epoch_loss = 0
      indices = np.arange(n_samples)
      if shuffle:
        rng.shuffle(indices)

      for idx in indices:
        x_j = self.inp.iloc[idx].values
        y_j = self.out.iloc[idx]
        fx = np.dot(self.weights, x_j)

        if y_j * fx <= 0:
          self.weights = self.weights + lr * y_j * x_j
          epoch_loss += 1

      self.loss_history.append(epoch_loss)
      if (i + 1) % 500 == 0 or i == 0:
        print(f"Epoch {i+1:04d} | Misclassifications: {epoch_loss}")

      if epoch_loss == 0:
        print(f"Converged early at epoch {i+1}!")
        break
    return self.weights

## 4. Custom Perceptron Training (With Shuffling)

In [ ]:
train_df = X_train.copy()
train_df['bias'] = 1.0
train_df['target'] = y_train.map(lambda val: -1 if val == 0 else 1)

test_df = X_test.copy()
test_df['bias'] = 1.0
test_df['target'] = y_test.map(lambda val: -1 if val == 0 else 1)

custom_clf_shuffled = CustomPerceptron(train_df)
print("--- Training Custom Perceptron with Shuffling ---")
final_weights_shuffled = custom_clf_shuffled.fit(epochs=5000, lr=5e-2, shuffle=True, random_state=42)

## 5. Performance Comparison & Evaluation

In [ ]:
X_eval = test_df.drop('target', axis=1)
y_true = test_df['target']

y_custom_pred_shuffled = []
for j in range(len(X_eval)):
    val = np.dot(final_weights_shuffled, X_eval.iloc[j].values)
    pred = 1 if val > 0 else -1
    y_custom_pred_shuffled.append(pred)

print("--- Shuffled Custom Perceptron Test Set Evaluation ---")
print(f"Accuracy: {accuracy_score(y_true, y_custom_pred_shuffled):.4f}\n")
print(classification_report(y_true, y_custom_pred_shuffled, target_names=['Class 0 (-1)', 'Class 1 (+1)']))

## 6. Visualizing the Decision Boundary Comparison

In [ ]:
def plot_boundary(weights, label, color, x_range):
    w1, w2, b = weights
    y_vals = -(w1 * x_range + b) / w2
    plt.plot(x_range, y_vals, label=label, color=color, linestyle='--', linewidth=2)

sklearn_weights = [clf.coef_[0][0], clf.coef_[0][1], clf.intercept_[0]]

plt.figure(figsize=(10, 6))
colors = test_df['target'].map({1: 'blue', -1: 'red'})
plt.scatter(test_df['x1'], test_df['x2'], c=colors, alpha=0.7, edgecolors='k', label='Test Data')

x_range = np.linspace(test_df['x1'].min() - 0.5, test_df['x1'].max() + 0.5, 100)
plot_boundary(final_weights_shuffled, label='Shuffled Custom Perceptron Boundary', color='darkorange', x_range=x_range)
plot_boundary(sklearn_weights, label='Sklearn SGDClassifier Boundary', color='green', x_range=x_range)

plt.title('Decision Boundary Comparison (Shuffled Custom Perceptron)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.axhline(0, color='grey', linewidth=0.5, linestyle=':')
plt.axvline(0, color='grey', linewidth=0.5, linestyle=':')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()